In [1]:
          # %%
# import Pkg; Pkg.add(["Turing", "Distributions", "LinearAlgebra", "Lux", "DifferentialEquations", "SciMLSensitivity", "Statistics", "DataFrames", "Plots", "Random", "ComponentArrays"])

using Turing, Distributions, LinearAlgebra
using Lux, SciMLSensitivity, ComponentArrays, StaticArrays, Zygote, DifferentialEquations
using Statistics, Random, Plots

┌ Warning: attempting to remove probably stale pidfile
│   path = "C:\\Users\\nirbh\\.julia\\compiled\\v1.12\\DifferentialEquations\\UQdwS_VxQKC.ji.pidfile"
└ @ FileWatching.Pidfile C:\Users\nirbh\.julia\juliaup\julia-1.12.5+0.x64.w64.mingw32\share\julia\stdlib\v1.12\FileWatching\src\pidfile.jl:247
[ Info: Precompiling DifferentialEquations [0c46a032-eb83-5123-abaf-570d42b7fbaa] (cache misses: wrong dep version loaded (8))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
┌ Warning: attempting to remove probably stale pidfile
│   path = "C:\\Users\\nirbh\\.julia\\compiled\\v1.12\\BoundaryValueDiffEq\\Levrg_VxQKC.ji.pidfile"
└ @ FileWatchi

In [ ]:
using CSV
using DataFrames

# 1. Define the file path
file_path = "e:/Neural_Spiking_Dynamics/notebooks/1_data_generation/single_spike_noisy_data.csv"

# 2. Read the CSV directly into a DataFrame (Raw Order: timestamp, V, m, n, h)
HH_data_raw = CSV.read(file_path, DataFrame)

# 3. Select all rows (:) and reorder the columns as desired
df_ordered = HH_data_raw[:, [:timestamp, :V, :n, :m, :h]]

# Display the first few rows to verify the new order
first(df_ordered, 5)
          

In [ ]:
# %%
df_ordered = HH_data[:, [:timestamp, :V, :n, :m, :h]]

# %%
t_train = Float32.(df_ordered.timestamp)
z_train = Float32.(Matrix(df_ordered[:, [:V, :n, :m, :h]])')

In [ ]:

# %%
nn = Lux.Chain(
    Lux.Dense(4 => 16, Lux.tanh), # 4 inputs, NO TIME
    Lux.Dense(16 => 16, Lux.tanh),
    Lux.Dense(16 => 4)            # 4 outputs
)

rng = Random.default_rng()
ps, st = Lux.setup(rng, nn) 

# Flatten parameters for SciML and Turing
p_initial = ComponentArray(ps)
axes_p = getaxes(p_initial) # Save the axes so we can rebuild the ComponentArray inside Turing

In [ ]:
# %%
# Float32 Rate Functions
α_m(V) = 0.1f0 .* (V .+ 40.0f0) ./ (1.0f0 .- exp.(-(V .+ 40.0f0) ./ 10.0f0))
β_m(V) = 4.0f0 .* exp.(-(V .+ 65.0f0) ./ 18.0f0)
α_h(V) = 0.07f0 .* exp.(-(V .+ 65.0f0) ./ 20.0f0)
β_h(V) = 1.0f0 ./ (1.0f0 .+ exp.(-(V .+ 35.0f0) ./ 10.0f0))
α_n(V) = 0.01f0 .* (V .+ 55.0f0) ./ (1.0f0 .- exp.(-(V .+ 55.0f0) ./ 10.0f0))
β_n(V) = 0.125f0 .* exp.(-(V .+ 65.0f0) ./ 80.0f0)

function hh_equations(u)
    V = u[1:1, :] 
    n = u[2:2, :]
    m = u[3:3, :]
    h = u[4:4, :]
    
    I_ext = 10.0f0
    g_Na  = 120.0f0
    g_K   = 36.0f0
    g_L   = 0.3f0
    E_Na  = 50.0f0
    E_K   = -77.0f0
    E_L   = -54.4f0
    C_m   = 1.0f0

    I_Na = g_Na .* (m.^3) .* h .* (V .- E_Na)
    I_K  = g_K .* (n.^4) .* (V .- E_K)
    I_L  = g_L .* (V .- E_L)
    
    dV = (I_ext .- I_Na .- I_K .- I_L) ./ C_m
    dn = α_n.(V) .* (1.0f0 .- n) .- β_n.(V) .* n
    dm = α_m.(V) .* (1.0f0 .- m) .- β_m.(V) .* m
    dh = α_h.(V) .* (1.0f0 .- h) .- β_h.(V) .* h
    
    return vcat(dV, dn, dm, dh)
end

In [ ]:
# %%
function neural_dynamics(u, p, t)
    input_2d = reshape(u, :, 1)
    dudt_2d, _ = nn(input_2d, p, st) 
    return vec(dudt_2d)
end

u0_cpu = Float32[-65.0f0, 0.05f0, 0.6f0, 0.32f0]
tspan = (0.0f0, 50.0f0)

node_prob = ODEProblem(neural_dynamics, u0_cpu, tspan, p_initial)

In [ ]:
# %%
@model function bayesian_pinode(t_train, z_train, prob, st, axes_p)
    # --- 1. Priors (The Latent Variables) ---
    
    # Neural ODE Weights: Normal distribution acting as L2 regularization
    theta ~ MvNormal(Zeros(length(prob.p)), 1.0 * I)
    
    # Scale-Aware Parameters: Half-Cauchy distributions for V, n, m, h
    # This automatically discovers if a variable is stiff and needs scaling
    scales ~ filldist(truncated(Cauchy(0.0, 1.0), lower=0.01), 4)
    
    # Physics Weight: Automatically balances data vs physics
    lambda_phys ~ Gamma(2.0, 0.5)
    
    # Biological Noise / Measurement Error
    sigma_obs ~ truncated(Normal(0.0, 0.5), lower=0.01)

    # --- 2. ODE Solve ---
    # Reconstruct the Lux parameters from the flat theta vector
    p_sampled = ComponentArray(theta, axes_p)
    new_prob = remake(prob, p=p_sampled)
    
    # Using Heun with InterpolatingAdjoint to handle stiff spikes without collapsing[cite: 1]
    sol = solve(new_prob, Heun(), saveat=t_train, reltol=1e-5, abstol=1e-5, sensealg=InterpolatingAdjoint())
    
    # If the solver fails (diverges), explicitly reject these parameters
    if sol.retcode != ReturnCode.Success
        Turing.@addlogprob! -Inf
        return
    end
    
    pred_z = Array(sol)
    
    # --- 3. Data Likelihood (Measurement Model) ---
    for i in 1:size(z_train, 2)
        for j in 1:4
            z_train[j, i] ~ Normal(pred_z[j, i], sigma_obs)
        end
    end

    # --- 4. Scale-Aware Physics Likelihood (Virtual Observation) ---
    # Forward pass on the predictions to get the learned vector field
    pred_derivs, _ = nn(pred_z, p_sampled, st)
    
    # Calculate the ground truth physics on those predictions
    true_derivs = hh_equations(pred_z)
    
    # Calculate Residuals
    residuals = pred_derivs .- true_derivs
    
    # We "observe" that the residual MUST be 0. 
    # The variance is dynamically managed by our sampled scales and lambda.
    for i in 1:size(residuals, 2)
        for j in 1:4
            0.0 ~ Normal(residuals[j, i], scales[j] / lambda_phys)
        end
    end
end

In [ ]:
# %%
# Instantiate the model
prob_model = bayesian_pinode(t_train, z_train, node_prob, st, axes_p)

println("Starting NUTS MCMC Sampling...")
# We use NUTS (No-U-Turn Sampler), which utilizes Zygote/SciML gradients
# 500 warmup steps to find the typical set, then draw 500 valid samples
chain = sample(prob_model, NUTS(0.65), 1000)

println("Bayesian Inference Complete!")

In [ ]:
# %%
# Plot the learned latent variables
# You will see the model "discovered" different scales for V vs n, m, h
plot(chain[["scales[1]", "scales[2]", "scales[3]", "scales[4]"]], 
     title="Posterior Distribution of Scale Factors (s_j)")

plot(chain[["lambda_phys", "sigma_obs"]], 
     title="Posterior of Physics Weight & Data Noise")